# TP L20 — Retour, Q, avantage et erreur TD

**Public :** débutant en apprentissage par renforcement · **Durée :** 45 à 60 min

Ce TP transforme les formules du chapitre en nombres vérifiables sur un choix de relais. Il est inspiré par la progression expérimentale de [FIDLE](https://gricad-gitlab.univ-grenoble-alpes.fr/talks/fidle/-/tree/master/DRL.PyTorch), sans reprise de son code.

À la fin, vous saurez distinguer récompense immédiate, retour actualisé, valeur d'action Q, valeur V, avantage et erreur temporelle TD.

## 1. Le scénario

Un routeur hésite entre deux relais. A donne un gain immédiat, puis provoque un débordement ; B paraît moins bon au premier pas, mais préserve la file.

- A : récompenses `(1, 0.5, -4)`
- B : récompenses `(0.2, 0.8, 1)`
- facteur d'actualisation : γ = 0,9

Le **retour** est la somme des récompenses futures pondérées : `G = r1 + γ r2 + γ² r3 + …`.

In [ ]:
from pathlib import Path
import sys

LAB_ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "return_td_lab.py").is_file()
)
sys.path.insert(0, str(LAB_ROOT))

from return_td_lab import (
    action_values, advantages, discounted_return, guided_summary,
    policy_value, td_error,
)

## 2. Du retour à Q, V et l'avantage

`Q(s,a)` mesure le retour attendu après l'action `a`. `V(s)` moyenne les Q selon la politique. L'**avantage** `A(s,a)=Q(s,a)-V(s)` dit si une action est meilleure que le comportement moyen de cette politique.

In [ ]:
trajectoires = {"A": (1.0, 0.5, -4.0), "B": (0.2, 0.8, 1.0)}
gamma = 0.9
q = action_values(trajectoires, gamma)
v = policy_value(q, {"A": 0.5, "B": 0.5})
adv = advantages(q, v)

print(f"Q(A) = {q['A']:.2f} ; Q(B) = {q['B']:.2f}")
print(f"V(s)  = {v:.2f}")
print(f"A(A)  = {adv['A']:.2f} ; A(B) = {adv['B']:.2f}")

**Interprétation attendue.** La récompense immédiate favorise A, mais son retour est négatif. Le RL devient utile précisément quand l'action modifie des conséquences futures que la décision myope ignore.

## 3. Sensibilité à γ

Un γ faible privilégie le prochain pas ; un γ élevé donne davantage de poids au débordement futur. Repérez le moment où le classement change.

In [ ]:
for gamma_test in (0.0, 0.3, 0.6, 0.9, 1.0):
    qa = discounted_return(trajectoires["A"], gamma_test)
    qb = discounted_return(trajectoires["B"], gamma_test)
    choix = "A" if qa > qb else "B"
    print(f"γ={gamma_test:.1f} : Q(A)={qa:5.2f}, Q(B)={qb:5.2f} → {choix}")

## 4. Erreur TD, terminaison et troncature

L'erreur TD compare une cible à la valeur courante : `δ = r + γ V(s') - V(s)`. Le terme futur disparaît lors d'une vraie **terminaison**. Une **troncature** (par exemple la limite de temps du simulateur) n'affirme pas que l'état terminal du problème a été atteint : il faut donc conserver le bootstrap.

In [ ]:
cas_en_cours = td_error(
    reward=1.0, gamma=0.9, next_value=2.0, current_value=0.5, terminated=False
)
cas_termine = td_error(
    reward=1.0, gamma=0.9, next_value=2.0, current_value=0.5, terminated=True
)
print(f"Transition en cours ou tronquée : δ={cas_en_cours:.2f}")
print(f"Transition réellement terminée   : δ={cas_termine:.2f}")

## 5. À vous

1. Remplacez la dernière récompense de A par `-1`. Le choix change-t-il ?
2. Donnez 80 % de probabilité à A dans la politique. Comment V et les avantages changent-ils ?
3. Expliquez en une phrase pourquoi une pénalité n'est pas une contrainte de sûreté.
4. Proposez une règle dure qui interdirait un relais indisponible avant même l'appel à la politique.

## 6. Auto-vérification

Ces assertions ne prouvent pas qu'une politique est sûre ; elles vérifient seulement les calculs élémentaires du TP.

In [ ]:
resume = guided_summary(gamma=0.9)
assert abs(discounted_return((1.0, 0.5, -4.0), 0.9) + 1.79) < 1e-12
assert resume["q_values"]["B"] > resume["q_values"]["A"]
assert cas_en_cours == 2.3
assert cas_termine == 0.5
print("Calculs validés. Expliquez maintenant leur sens, pas seulement leur valeur.")